# AlphaTrain Pillar 2M: Self-Play Iteration 2 (Higher-IQ Teacher)

Second self-play loop with stronger data:
- **800 sims** self-play (was 400 in 2L) — deeper search, better moves
- **80/20 mix** (was 70/30) — less tactical dilution
- **1,500 games, 1.25M states** from 2L model (mean score 1,754)

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `expert_v2_pairwise_g095_surv1.0_ms30.pt.gz` — expert data (on Drive)
3. `selfplay_v2_surv_ms30.pt.gz` — self-play v2 data (NEW)
4. `pillar2l_best.pt` — base model (on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code (always fresh)
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy model
os.makedirs('/content/alphatrain/data', exist_ok=True)
print('Copying model...')
shutil.copy(f'{DRIVE}/pillar2l_best.pt', '/content/alphatrain/data/model.pt')

# Decompress expert data
print('Decompressing expert data...')
t0 = time.time()
!gzip -dc {DRIVE}/expert_v2_pairwise_g095_surv1.0_ms30.pt.gz > /content/alphatrain/data/expert.pt
print(f'Expert: {os.path.getsize("/content/alphatrain/data/expert.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

# Decompress self-play v2 data
print('Decompressing self-play v2 data...')
t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v2_surv_ms30.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Self-play: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

for name, path in [('Expert', '/content/alphatrain/data/expert.pt'),
                    ('Self-play', '/content/alphatrain/data/selfplay.pt')]:
    d = torch.load(path, weights_only=True, map_location='cpu')
    print(f'\n{name}: {d["boards"].shape[0]:,} states, '
          f'max_score={d["max_score"]}, bins={d["num_value_bins"]}')
    del d

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2M: 80% expert + 20% self-play v2 (800-sim games)
# Warm start from 2L (best self-play model)
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/expert.pt \
    --selfplay-tensor alphatrain/data/selfplay.pt \
    --selfplay-fraction 0.2 \
    --gpu-data --amp --compile \
    --value-bins 128 --value-channels 32 --value-hidden 512 --value-dropout 0.3 \
    --val-weight 0.01 --rank-weight 1.0 \
    --endgame-fraction 0.3 --endgame-threshold 100 \
    --adversarial-ranking \
    --resume alphatrain/data/model.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2m_best.pt \
    --save-dir /content/checkpoints/pillar2m

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2m/{f}'
    dst = f'{DRIVE}/pillar2m_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')